# Setup - Requires a docker container yuh
```bash
sudo docker run --device=nvidia.com/gpu=all \
                                                      -p 51000:51000 \
                                                      --shm-size 2g \
                                                      -e CLIP_MODEL_NAME=jinaai/jina-clip-v2 \
                                                      -v ~/.cache/huggingface:/root/.cache/huggingface \
                                                      jinaai/clip-server:latest
                                                      
```

In [46]:
import nest_asyncio
nest_asyncio.apply()
import os
import glob
import io
from PIL import Image, UnidentifiedImageError
from docarray import Document, DocumentArray
from clip_client import Client
from dotenv import load_dotenv
from neo4j import GraphDatabase
import numpy as np
import os
from tqdm import tqdm

c = Client('grpc://172.17.0.8:51000') 

c.profile()

env_path = os.path.expanduser("~/workspace/.env")
load_dotenv(env_path, override=True)
uri = os.getenv("NEO4J_URI")
print(f"Loading from: {env_path}")
print(f"NEO4J_URI found: {uri}") 
if not uri:
    raise ValueError("NEO4J_URI is missing! Check the .env file path.")

driver = GraphDatabase.driver(
    uri,
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
)

Roundtrip  5ms  100% 
├──  Client-server network  4ms  79% 
└──  Server  1ms  21% 
    ├──  Gateway-CLIP network  0ms  0% 
    └──  CLIP model  1ms  100%

Loading from: /home/jovyan/workspace/.env
NEO4J_URI found: neo4j+s://75d48924.databases.neo4j.io


In [47]:
image_dir = os.path.expanduser('~/workspace/Datasets/Pictures/')
image_paths = glob.glob(os.path.join(image_dir, '*')) 
valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
image_paths = [p for p in image_paths if os.path.splitext(p)[1].lower() in valid_extensions]
print(f"Found {len(image_paths)} images in {image_dir}")

Found 1895 images in /home/jovyan/workspace/Datasets/Pictures/


In [48]:
docs = DocumentArray([Document(uri=path) for path in image_paths])

def data_generator(docs):
    for doc in docs:
        try:
            doc.load_uri_to_blob() 
            with Image.open(io.BytesIO(doc.blob)) as img:
                img.verify() 
            yield doc
            
        except (UnidentifiedImageError, OSError, Exception) as e:
            print(f"Skipping corrupt file: {doc.uri} | Error: {e}")

In [49]:
CACHE_DIR  = os.path.expanduser("~/workspace/.cache")
CACHE_FILE = os.path.join(CACHE_DIR, "clip_embeddings_v2.pkl")

os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Cache file: {CACHE_FILE}")

# Initialize variable to track if we found a VALID cache
valid_cache_found = False

if os.path.exists(CACHE_FILE):
    print("Found cache file. Verifying consistency...")
    try:
        with open(CACHE_FILE, 'rb') as f:
            cache = pickle.load(f)
            
        cached_emb = cache['embeddings']
        
        # --- THE FIX IS HERE ---
        # Check if cache size matches current document count
        if len(cached_emb) == len(docs):
            embeddings = SimpleNamespace(embeddings=cached_emb)
            print(f"✔ Cache valid! Loaded {embeddings.embeddings.shape} embeddings "
                  f"(generated {cache.get('timestamp', 'unknown')})")
            valid_cache_found = True
        else:
            print(f"✖ Cache mismatch: Cache has {len(cached_emb)}, but docs have {len(docs)}.")
            print("  Invalidating cache to force regeneration...")
            
    except Exception as e:
        print(f"Error reading cache: {e}. Regenerating")

# If no cache, or if cache was invalid/stale, generate now
if not valid_cache_found:
    print(f"Encoding {len(docs)} images with CLIP")
    
    # Assuming 'c' is your CLIP client/model and data_generator is defined
    embeddings = c.encode(
        data_generator(docs),
        batch_size=16,
        show_progress=True
    )
    print(f"Encoding complete! Shape: {embeddings.embeddings.shape}")
    
    cache_data = {
        'embeddings': embeddings.embeddings,
        'timestamp': datetime.now().isoformat(),
        'num_images': len(docs)
    }
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(cache_data, f)
    print(f"Embeddings cached forever at:\n   {CACHE_FILE}")

print(f"\nReady! Embeddings shape: {embeddings.embeddings.shape}")
print("You can now run Neo4j upload, similarity search, etc. — instantly!")

Output()

Cache file: /home/jovyan/workspace/.cache/clip_embeddings_v2.pkl
Found cache file. Verifying consistency...
✖ Cache mismatch: Cache has 1872, but docs have 1895.
  Invalidating cache to force regeneration...
Encoding 1895 images with CLIP


Encoding complete! Shape: (1895, 512)
Embeddings cached forever at:
   /home/jovyan/workspace/.cache/clip_embeddings_v2.pkl

Ready! Embeddings shape: (1895, 512)
You can now run Neo4j upload, similarity search, etc. — instantly!


In [ ]:

def create_index(tx):
    tx.run("""
        CREATE VECTOR INDEX image_clip_embedding IF NOT EXISTS
        FOR (i:Image) ON (i.embedding)
        OPTIONS {
            indexConfig: {
                `vector.dimensions`: 512,
                `vector.similarity_function`: 'cosine'
            }
        }
    """)

BATCH_SIZE = 500

def upsert_batch(tx, batch):
    tx.run("""
        UNWIND $batch AS row
        MERGE (i:Image {uri: row.uri})
        SET i.filename = row.filename,
            i.embedding = row.embedding,
            i.cached_at = datetime()
    """, batch=batch)

print("Starting migration of cached embeddings → Neo4j...")

with driver.session() as session:
    session.execute_write(create_index)
    print("Vector index ready")

    batch = []
    total = len(docs)
    
    for idx, doc in enumerate(tqdm(docs, desc="Uploading to Neo4j", unit="img")):
        uri = getattr(doc, 'uri', None)
        if not uri:
            continue  # skip any broken docs
            
        filename = os.path.basename(uri)
        
        batch.append({
            "uri": uri,
            "filename": filename,
            "embedding": embeddings.embeddings[idx].tolist()  # ← from cache!
        })
        
        if len(batch) >= BATCH_SIZE:
            session.execute_write(upsert_batch, batch)
            batch.clear()
    
    if batch:
        session.execute_write(upsert_batch, batch)

driver.close()
print(f"\nSUCCESS! {total} images are now permanently stored in Neo4j")
print("From now on: instant startup + sub-50ms similarity search")

In [ ]:
uri = os.getenv("NEO4J_URI")

driver = GraphDatabase.driver(
    uri,
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
)

control_dir = os.path.expanduser('~/workspace/Datasets/Pictures/Control/')
control_paths = sorted([
    os.path.join(control_dir, f)
    for f in os.listdir(control_dir)
    if not f.startswith('.')
])

TOP_K = 5
print(f"Running clean similarity search for {len(control_paths)} control images...\n")

for query_path in control_paths:
    query_name = os.path.basename(query_path)
    print(f"Query → {query_name}")

    # Encode query (works with both clip-client versions)
    result = c.encode([query_path])
    query_vector = result[0].tolist() if isinstance(result, np.ndarray) else result.embeddings[0].tolist()

    # Search Neo4j
    with driver.session() as session:
        raw_results = session.run("""
            CALL db.index.vector.queryNodes('image_clip_embedding', 50, $vec)
            YIELD node, score
            RETURN node.uri AS uri, score
            ORDER BY score DESC
        """, vec=query_vector).data()

    # REMOVE SELF-MATCHES & NEAR-DUPLICATES
    clean_matches = []
    for r in raw_results:
        if r['uri'] == query_path:          # exact same file
            continue
        if r['score'] > 0.999:              # near-perfect duplicate
            continue
        clean_matches.append(r)
        if len(clean_matches) >= TOP_K:
            break

    # Fallback: if nothing found, show top non-exact matches
    if not clean_matches:
        clean_matches = [r for r in raw_results[:TOP_K] if r['uri'] != query_path]

    # Debug print
    print(f"\n{'='*85}")
    print(f"QUERY: {query_name}")
    print(f"{'='*85}")
    for i, m in enumerate(clean_matches):
        print(f"{i+1}. {m['score']:.4f} → {os.path.basename(m['uri'])}")

    # Visual grid
    cols = len(clean_matches) + 1
    fig, axes = plt.subplots(1, cols, figsize=(4.2 * cols, 5))
    if cols == 1:
        axes = [axes]

    # Query image
    try:
        img = mpimg.imread(query_path)
        axes[0].imshow(img)
        axes[0].set_title(f"QUERY\n{query_name}", color='blue', fontsize=14, fontweight='bold')
    except Exception as e:
        axes[0].text(0.5, 0.5, f"ERROR\n{e}", color='red', ha='center')
    axes[0].axis('off')

    # Real matches
    for i, m in enumerate(clean_matches):
        ax = axes[i+1]
        try:
            img = mpimg.imread(m['uri'])
            ax.imshow(img)
            score = m['score']
            color = 'green' if score >= 0.90 else ('orange' if score >= 0.80 else 'black')
            ax.set_title(f"#{i+1} ({score:.3f})\n{os.path.basename(m['uri'])}",
                        color=color, fontsize=10)
        except:
            ax.text(0.5, 0.5, "NO IMG", color='red', ha='center')
        ax.axis('off')

    plt.tight_layout()
    plt.show()

driver.close()
print("\nDONE — Clean, beautiful, no self-matches. Perfect.")